# Génération de population — pipeline multi-taille avec checkpoints

Ce notebook orchestre la création de populations synthétiques pour plusieurs tailles cibles,
avec checkpoints intermédiaires dans un dossier `Temp/` situé à côté du notebook.

## Paramètres

- `POPULATION_SIZES` : liste des tailles à générer (multiples de 100)
- `FORCE_REGENERATE` : force la régénération depuis eqasim (ignore le cache de l'API)
- `FORCE_STEP` : force la reprise à partir d'une étape — `raw`, `fixed`, `pt_enriched`, `routed` ou `scheduled`

## Pipeline avec checkpoints

| Étape | Entrée | Sortie |
|---|---|---|
| 1 – Génération eqasim | API eqasim | `Temp/1_raw/` |
| 2 – Validation activités | `Temp/1_raw/` | `Temp/2_fixed/` |
| 3 – Enrichissement PT | `Temp/2_fixed/` | `Temp/3_pt_enriched/` |
| 4 – Calcul itinéraires OSMnx | `Temp/3_pt_enriched/` | `Temp/4_routed/` |
| 5 – Ajustement horaires | `Temp/4_routed/` | `Temp/5_scheduled/` |
| Export final | `Temp/5_scheduled/` | `data/eqasim_output/` |

À chaque étape, si le fichier de sortie existe déjà dans `Temp/`, l'étape est ignorée.
Pour forcer la reprise à une étape donnée (et toutes les suivantes), définir `FORCE_STEP`.

## Prérequis

| Service | Commande | Port |
|---|---|---|
| eqasim | `docker compose up eqasim` | 8003 |

Dépendances Python : `numpy`, `pandas`, `tqdm`, `osmnx`

In [ ]:
# ── Paramètres ────────────────────────────────────────────────────────────────
POPULATION_SIZES      = [100, 200, 300, 500]   # multiples de 100 ; peut atteindre des centaines de milliers
GENERATE_PERSONALITY  = False              # True → génère les Big Five (lent)
FORCE_REGENERATE      = False              # True → appelle eqasim avec force=True + reécrit Temp/raw/
FORCE_STEP            = None              # None | 'raw' | 'fixed' | 'pt_enriched' | 'routed' | 'scheduled'
                                           # Force la reprise à partir de cette étape (et toutes les suivantes)

BBOX = None                               # [min_lon, min_lat, max_lon, max_lat] WGS84 — None = dept 31
# BBOX = [1.35, 43.55, 1.50, 43.65]      # exemple : centre de Toulouse

CLEAR_DOWNSTREAM_ON_REGENERATE = True    # True → supprime les checkpoints downstream quand l'étape 1 régénère un raw

EQASIM_URL = 'http://localhost:8003'

## Initialisation de l'environnement

Création de l'arborescence de dossiers temporaires (`Temp/`) et définition des chemins vers les données sources et de sortie. Chaque étape écrit dans un sous-dossier numéroté (`1_raw/`, `2_fixed/`, …) pour permettre la reprise depuis n'importe quel point sans tout recalculer.

La fonction `should_force(step)` détermine si une étape doit être rejouée en tenant compte de `FORCE_REGENERATE` et `FORCE_STEP`.

In [ ]:
# ── Chemins & dossiers Temp ───────────────────────────────────────────────────
import json
import os
import sys
import time
import urllib.request
import urllib.error
from pathlib import Path

REPO_ROOT    = Path('../../../').resolve()
POP_DIR      = REPO_ROOT / 'data' / 'eqasim_output'
NOTEBOOK_DIR = Path('.').resolve()
TEMP_DIR     = NOTEBOOK_DIR / 'Temp'

TEMP_RAW       = TEMP_DIR / '1_raw'
TEMP_FIXED     = TEMP_DIR / '2_fixed'
TEMP_PT        = TEMP_DIR / '3_pt_enriched'
TEMP_ROUTED    = TEMP_DIR / '4_routed'
TEMP_SCHEDULED = TEMP_DIR / '5_scheduled'

for d in [POP_DIR, TEMP_RAW, TEMP_FIXED, TEMP_PT, TEMP_ROUTED, TEMP_SCHEDULED]:
    d.mkdir(parents=True, exist_ok=True)

# ── Utilitaires ───────────────────────────────────────────────────────────────
STEPS = ['raw', 'fixed', 'pt_enriched', 'routed', 'scheduled']

def should_force(step_name: str) -> bool:
    if FORCE_REGENERATE and step_name == 'raw':
        return True
    if FORCE_STEP is None or FORCE_STEP not in STEPS:
        return False
    return STEPS.index(step_name) >= STEPS.index(FORCE_STEP)

def pop_filename(n: int) -> str:
    return f'toulouse_population_{n}.json'

def save_json(data, path: Path) -> None:
    tmp = path.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.rename(tmp, path)

def load_json(path: Path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def check_temporal_order(data: list) -> tuple[int, int]:
    """Check start_time ordering (at most 1 decreasing gap allowed for midnight wrap).
    Returns (n_persons_with_error, n_total_errors).
    """
    n_persons, n_total = 0, 0
    for person in data:
        acts = person.get('identity', {}).get('activities', [])
        n = len(acts)
        if n == 0:
            continue
        errors = []
        for i, act in enumerate(acts):
            s, e = act.get('start_time'), act.get('end_time')
            if s is not None and e is not None and (e - s) % 86400 == 0:
                errors.append(f"act[{i}] durée nulle")
        nb_ecarts = sum(
            1 for i in range(n)
            if acts[(i - 1) % n].get('start_time', 0) > acts[i].get('start_time', 0)
        )
        if nb_ecarts > 1:
            errors.append(f"{nb_ecarts} écarts décroissants sur start_time")
        if errors:
            n_persons += 1
            n_total += len(errors)
    return n_persons, n_total

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'POP_DIR   : {POP_DIR}')
print(f'TEMP_DIR  : {TEMP_DIR}')
print(f'Tailles   : {POPULATION_SIZES}')

## Chargement des dépendances scientifiques

Import des bibliothèques de calcul numérique (`numpy`, `pandas`) et configuration des constantes partagées entre les étapes :

| Constante | Valeur | Rôle |
|---|---|---|
| `GTFS_STOPS` | `data/gtfs/tisseo_gtfs/stops.txt` | Arrêts Tisséo pour l'enrichissement TC |
| `OSMNX_CACHE` | `data/osmnx_cache/` | Cache des graphes routiers (évite le re-téléchargement) |
| `MAX_WORKERS` | 12 | Parallélisme du calcul de routes |
| `MAX_PT_DIST_M` | 1 500 m | Rayon de rattachement à un arrêt TC |

In [ ]:
# ── Imports scientifiques & constantes ───────────────────────────────────────
import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

GTFS_STOPS     = REPO_ROOT / 'data' / 'gtfs' / 'tisseo_gtfs' / 'stops.txt'
OSMNX_CACHE    = REPO_ROOT / 'data' / 'osmnx_cache'
SCRIPTS_POP    = REPO_ROOT / 'scripts' / 'data' / 'population'
LLMAGENTS_PATH = str(REPO_ROOT / 'llm-agents')

CACHE_KEY     = hashlib.md5(b'Toulouse, France_30000').hexdigest()[:12]
MAX_WORKERS   = 12
MAX_PT_DIST_M = 1500.0

if LLMAGENTS_PATH not in sys.path:
    sys.path.insert(0, LLMAGENTS_PATH)
if str(SCRIPTS_POP) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_POP))

from population_utils import TRIP_MODES

print(f'GTFS_STOPS:  {GTFS_STOPS}')
print(f'OSMNX_CACHE: {OSMNX_CACHE}')
print(f'OSMnx cache key: {CACHE_KEY}')

## Vérification du service eqasim

Avant tout traitement, on s'assure que le service de génération de population eqasim répond sur `EQASIM_URL` (port 8003). En cas d'échec après 3 tentatives, le pipeline s'arrête immédiatement avec un message d'erreur explicite.

> **Prérequis** : `docker compose up eqasim` doit être lancé avant d'exécuter cette cellule.

In [ ]:
# ── Vérification santé du service eqasim ─────────────────────────────────────
def check_health(url: str, retries: int = 3, delay: float = 2.0) -> bool:
    for i in range(retries):
        try:
            with urllib.request.urlopen(f'{url}/health', timeout=5) as r:
                return r.status == 200
        except Exception as e:
            print(f'  tentative {i+1}/{retries} : {e}')
            if i < retries - 1:
                time.sleep(delay)
    return False

if check_health(EQASIM_URL):
    print(f'Service eqasim OK → {EQASIM_URL}')
else:
    raise RuntimeError(
        f'Service eqasim inaccessible sur {EQASIM_URL}. '
        'Démarrez le service avec : docker compose up eqasim'
    )

---
## Étape 1 — Génération eqasim → `Temp/1_raw/`

Appel à l'API eqasim pour chaque taille de population définie dans `POPULATION_SIZES`. eqasim synthétise des agents toulousains avec leurs activités quotidiennes (domicile, travail, loisirs…) à partir de données INSEE et d'enquêtes de déplacements (EMC²).

- **Cache** : si le fichier existe déjà dans `Temp/1_raw/` et que `FORCE_REGENERATE=False`, l'appel API est ignoré.
- **Sortie** : un fichier JSON par taille contenant la liste des personnes avec leurs activités brutes et leurs attributs socio-démographiques.
- **Validation** : après génération, l'ordre temporel des activités est vérifié — au plus un écart décroissant est toléré (passage minuit).

In [ ]:
# ── Étape 1 — Génération eqasim → Temp/1_raw/ ────────────────────────────────
print('=' * 60)
print('ÉTAPE 1 — Génération eqasim → Temp/1_raw/')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname    = pop_filename(pop_size)
    raw_path = TEMP_RAW / fname

    if not should_force('raw') and raw_path.exists():
        size_mb = raw_path.stat().st_size / 1_048_576
        print(f'[SKIP] {fname}  ({size_mb:.1f} Mo) — déjà dans Temp/1_raw/')
        continue

    payload = {
        'population_size':      pop_size,
        'generate_personality': GENERATE_PERSONALITY,
        'force':                FORCE_REGENERATE,
    }
    if BBOX is not None:
        payload['bbox'] = BBOX

    body = json.dumps(payload).encode()
    req  = urllib.request.Request(
        f'{EQASIM_URL}/generate',
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )

    print(f'[GEN]  {fname}  (population_size={pop_size})…')
    t0 = time.monotonic()

    try:
        with urllib.request.urlopen(req, timeout=7200) as resp:
            result = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        result = json.loads(e.read())
        raise RuntimeError(f'eqasim HTTP {e.code} : {result}')

    elapsed = time.monotonic() - t0
    if result.get('status') != 'ok':
        raise RuntimeError(f'Échec eqasim pour {pop_size} agents : {result}')

    eqasim_out = POP_DIR / fname
    if not eqasim_out.exists():
        raise FileNotFoundError(f'Fichier eqasim introuvable : {eqasim_out}')

    data = load_json(eqasim_out)
    save_json(data, raw_path)
    size_mb = raw_path.stat().st_size / 1_048_576
    print(f'       {len(data)} personnes en {elapsed:.1f}s  ({size_mb:.1f} Mo) → Temp/1_raw/{fname}')
    if CLEAR_DOWNSTREAM_ON_REGENERATE:
        for _dl_dir in [TEMP_FIXED, TEMP_PT, TEMP_ROUTED, TEMP_SCHEDULED]:
            _p = _dl_dir / fname
            if _p.exists():
                _p.unlink()
                print(f'       [CASCADE] Supprimé {_dl_dir.name}/{fname}')
        _sqlite = OSMNX_CACHE / f'toulouse_population_{pop_size}' / 'osmnx_cache.db'
        if _sqlite.exists():
            _sqlite.unlink()
            print(f'       [CASCADE] Supprimé cache SQLite OSMnx : {_sqlite.relative_to(REPO_ROOT)}')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 1 terminée.')

---
## Étape 2 — Validation & correction des activités → `Temp/2_fixed/`

Les données brutes eqasim peuvent contenir des séquences d'activités invalides (chevauchements, durées nulles, enchaînements incohérents). Cette étape applique deux passes de nettoyage :

1. **Correction des violations de séquence** via `fix_activities` : réajustement des bornes temporelles pour éliminer chevauchements et durées négatives.
2. **Fusion des activités redondantes** : deux activités consécutives (ou circulaires) de même `purpose` et même localisation sont fusionnées en une seule, pour éviter des micro-trajets de durée nulle.

Chaque agent est re-validé après correction ; les cas encore invalides sont signalés mais n'interrompent pas le pipeline.

In [ ]:
# ── Étape 2 — Validation & correction des activités → Temp/2_fixed/ ──────────
from population_utils import check_activities, fix_activities

print('=' * 60)
print('ÉTAPE 2 — Validation & correction des activités → Temp/2_fixed/')
print('=' * 60)

def _same_loc(a, b):
    la, lb = a.get('location'), b.get('location')
    if la is None or lb is None:
        return la is lb
    return la.get('lon') == lb.get('lon') and la.get('lat') == lb.get('lat')

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    raw_path   = TEMP_RAW   / fname
    fixed_path = TEMP_FIXED / fname

    if not should_force('fixed') and fixed_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/2_fixed/')
        continue

    if not raw_path.exists():
        raise FileNotFoundError(f'Fichier raw manquant — relancer étape 1 : {raw_path}')

    data = load_json(raw_path)

    # Correction des violations de séquence
    persons_fixed = 0
    fixed_data = []
    for person in data:
        fixed_person, _ = fix_activities(person)
        remaining = check_activities(fixed_person)
        if remaining:
            print(f'  !! person {person.get("person_id", "?")} : encore invalide après correction : {remaining}')
        fixed_data.append(fixed_person)
        persons_fixed += 1

    # Fusion : activités consécutives ET circulaires partageant même purpose ET même location
    n_merged = 0
    for person in fixed_data:
        acts = person.get('identity', {}).get('activities', [])

        # Fusion consécutive
        i = 0
        while i < len(acts) - 1:
            if acts[i].get('purpose') == acts[i + 1].get('purpose') and _same_loc(acts[i], acts[i + 1]):
                acts[i]['end_time'] = acts[i + 1]['end_time']
                acts[i]['scheduled_start_time'] = None
                acts.pop(i + 1)
                n_merged += 1
            else:
                i += 1

        # Fusion circulaire : première et dernière
        if len(acts) >= 2 and acts[0].get('purpose') == acts[-1].get('purpose') and _same_loc(acts[0], acts[-1]):
            last = acts.pop()
            acts[0]['start_time'] = last['start_time']
            acts[0]['scheduled_start_time'] = None
            n_merged += 1

    save_json(fixed_data, fixed_path)
    print(f'[OK]   {fname} — {persons_fixed} corrigé(s), {n_merged} fusion(s) → Temp/2_fixed/')
    n_p, n_e = check_temporal_order(fixed_data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(fixed_data) - n_p}/{len(fixed_data)} personnes valides')

print()
print('Étape 2 terminée.')

---
## Étape 3 — Enrichissement transports en commun → `Temp/3_pt_enriched/`

Pour chaque localisation d'activité, on calcule si elle se trouve à moins de `MAX_PT_DIST_M` (1 500 m) d'un arrêt Tisséo. Ce flag `public_transport` est ensuite utilisé par l'agent LLM pour décider du mode de déplacement pertinent (marche, voiture, TC…).

- **Source** : fichier `stops.txt` du GTFS Tisséo — 5 661 arrêts chargés en mémoire.
- **Méthode** : recherche du plus proche arrêt par distance euclidienne approximative sur les coordonnées WGS84.
- Le nombre de localisations enrichies est affiché par fichier.

In [ ]:
# ── Étape 3 — Enrichissement des flags public_transport → Temp/3_pt_enriched/ ─
from population_utils import enrich_public_transport

print('=' * 60)
print('ÉTAPE 3 — Enrichissement public_transport → Temp/3_pt_enriched/')
print('=' * 60)

stops_df  = pd.read_csv(GTFS_STOPS, usecols=['stop_lat', 'stop_lon'])
stop_lats = stops_df['stop_lat'].values
stop_lons = stops_df['stop_lon'].values
print(f'Chargement GTFS : {len(stops_df)} arrêts')
print()

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    fixed_path = TEMP_FIXED / fname
    pt_path    = TEMP_PT    / fname

    if not should_force('pt_enriched') and pt_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/3_pt_enriched/')
        continue

    if not fixed_path.exists():
        raise FileNotFoundError(f'Fichier fixed manquant — relancer étape 2 : {fixed_path}')

    data = load_json(fixed_path)
    n = enrich_public_transport(data, stop_lats, stop_lons, MAX_PT_DIST_M)
    save_json(data, pt_path)
    print(f'[OK]   {fname} — {n} localisation(s) enrichie(s) → Temp/3_pt_enriched/')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 3 terminée.')

---
## Étape 4 — Calcul des itinéraires OSMnx → `Temp/4_routed/`

Pour chaque paire (origine, destination, mode) présente dans la population, on calcule l'itinéraire réel sur le réseau routier ou piéton de Toulouse via **OSMnx**. Le calcul est massivement parallélisé sur `MAX_WORKERS` processus (`spawn`, évite les conflits GIL).

- **Déduplication globale** : les paires communes à plusieurs tailles de population ne sont calculées qu'une seule fois, puis réutilisées via `route_cache`.
- **Cache OSMnx** : le graphe du réseau est chargé une fois par worker depuis `OSMNX_CACHE/` (la clé de cache correspond à la zone Toulouse ~30 000 nœuds).
- **Résultat injecté** : chaque trajet reçoit une durée, une distance et la géométrie du chemin emprunté.
- **Cas inaccessibles** : si aucun chemin n'existe (zones déconnectées), la route reste `None` et est comptabilisée séparément.

In [ ]:
# ── Étape 4 — Calcul des itinéraires (OSMnx) → Temp/4_routed/ ────────────────
import multiprocessing
from datetime import datetime as _dt, date as _date

from population_utils import collect_pairs_for_file, apply_routes
from route_worker import init_worker, compute_route_worker

print('=' * 60)
print('ÉTAPE 4 — Calcul des itinéraires OSMnx → Temp/4_routed/')
print('=' * 60)

# Identifier les tailles qui ont besoin d'être routées
to_route = []
for pop_size in POPULATION_SIZES:
    fname       = pop_filename(pop_size)
    routed_path = TEMP_ROUTED / fname

    if not should_force('routed') and routed_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/4_routed/')
        continue

    pt_path = TEMP_PT / fname
    if not pt_path.exists():
        raise FileNotFoundError(f'Fichier pt_enriched manquant — relancer étape 3 : {pt_path}')

    to_route.append(pop_size)

if not to_route:
    print()
    print('Tous les fichiers sont déjà routés — étape 4 ignorée.')
else:
    # Chargement des données à router
    all_data_to_route: dict[int, list] = {}
    for pop_size in to_route:
        all_data_to_route[pop_size] = load_json(TEMP_PT / pop_filename(pop_size))

    # Collecte des paires manquantes, déduplication globale entre toutes les tailles
    # Clé : (lat1, lon1, lat2, lon2, mode, hour) — hour = heure de départ réelle pour car,
    # None pour foot/bicycle (résultat indépendant du temps → déduplication optimale).
    print()
    print('Collecte des paires origin-destination manquantes…')
    global_missing: set = set()
    per_size_missing: dict[int, set] = {}
    for pop_size, data in all_data_to_route.items():
        pairs = collect_pairs_for_file(data)
        per_size_missing[pop_size] = pairs
        global_missing |= pairs
        print(f'  toulouse_population_{pop_size} : {len(pairs)} paires manquantes')

    print(f'Total unique global : {len(global_missing)} paires à calculer')
    print()

    route_cache: dict[tuple, dict | None] = {}

    if global_missing:
        # Pour foot/bicycle, hour est None dans la clé → on passe 8 comme heure de calcul
        # (la congestion ne s'applique pas, le résultat est identique quelle que soit l'heure).
        tasks = [
            (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
            for lat1, lon1, lat2, lon2, mode, hour in global_missing
        ]

        print(f'Calcul de {len(tasks)} routes avec {MAX_WORKERS} workers…')
        t0 = time.monotonic()

        ctx = multiprocessing.get_context('spawn')
        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            mp_context=ctx,
            initializer=init_worker,
            initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
        ) as pool:
            results = list(tqdm(
                pool.map(compute_route_worker, tasks, chunksize=20),
                total=len(tasks),
                desc='routing',
            ))

        elapsed = time.monotonic() - t0
        print(f'Terminé en {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route en moyenne)')

        ok_count = null_count = 0
        for args, result in results:
            lat1, lon1, lat2, lon2, mode, actual_hour = args
            route_cache[(lat1, lon1, lat2, lon2, mode, actual_hour if mode == 'car' else None)] = result
            if result is None:
                null_count += 1
            else:
                ok_count += 1
        print(f'Routes : {ok_count} OK, {null_count} inaccessibles (None)')
        print()

    # Injection des routes et sauvegarde
    for pop_size in to_route:
        fname = pop_filename(pop_size)
        data  = all_data_to_route[pop_size]
        written = apply_routes(data, route_cache)
        save_json(data, TEMP_ROUTED / fname)
        print(f'[OK]   {fname} — {written} routes injectées → Temp/4_routed/')
        n_p, n_e = check_temporal_order(data)
        print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

    # Alimentation du cache SQLite OSMnx par jeu de test
    # Lundi de référence : seul le jour de semaine compte pour la congestion TomTom.
    from trip_helper.osmnx_persistent_cache import OsmnxPersistentCache
    _sim_date = _date(2024, 1, 8)  # lundi

    for pop_size in to_route:
        _cache_dir    = OSMNX_CACHE / f'toulouse_population_{pop_size}'
        _sqlite_cache = OsmnxPersistentCache(str(_cache_dir))
        _n_stored = 0
        for (_lat1, _lon1, _lat2, _lon2, _mode, _hour) in per_size_missing[pop_size]:
            _hour_key  = _hour if _mode == 'car' else None
            _result    = route_cache.get((_lat1, _lon1, _lat2, _lon2, _mode, _hour_key))
            _actual_h  = _hour if _hour is not None else 8
            _cdt       = _dt(_sim_date.year, _sim_date.month, _sim_date.day, _actual_h, 0)
            _p_key, _p_date, _p_dow, _p_bucket = OsmnxPersistentCache.make_key(
                _cdt, _mode, _lat1, _lon1, _lat2, _lon2
            )
            _sqlite_cache.store(_p_key, _p_date, _p_dow, _p_bucket, _mode,
                                _lat1, _lon1, _lat2, _lon2, _result)
            _n_stored += 1
        print(f'  SQLite OSMnx : {_n_stored} routes → {_cache_dir.relative_to(REPO_ROOT)}/osmnx_cache.db')

print()
print('Étape 4 terminée.')

---
## Étape 5 — Ajustement des horaires → `Temp/5_scheduled/`

Les itinéraires calculés à l'étape 4 ont des durées réelles qui peuvent créer des conflits dans l'agenda journalier de chaque agent. Cette étape recale les `scheduled_start_time` de chaque activité pour garantir la cohérence temporelle sur 24 h :

- Les activités sont décalées vers la gauche ou la droite pour absorber les temps de trajet réels.
- Les chevauchements sont résolus par compression des marges disponibles.
- Une **validation stricte** est appliquée en fin de traitement : tout ordre temporel invalide (plus d'un écart décroissant) lève une `ValueError` qui interrompt le pipeline — mieux vaut détecter l'incohérence ici que la propager dans la simulation GAMA.

In [ ]:
# ── Étape 5 — Ajustement des horaires → Temp/5_scheduled/ ────────────────────
from population_utils import ajuster_planning
print('=' * 60)
print('ÉTAPE 5 — Ajustement des horaires → Temp/5_scheduled/')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    routed_path    = TEMP_ROUTED    / fname
    scheduled_path = TEMP_SCHEDULED / fname

    if not should_force('scheduled') and scheduled_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/5_scheduled/')
        continue

    if not routed_path.exists():
        raise FileNotFoundError(f'Fichier routed manquant — relancer étape 4 : {routed_path}')

    data = load_json(routed_path)

    for entry in data:
        acts     = entry.get('identity', {}).get('activities', [])
        adjusted = ajuster_planning(fname,entry.get('person_id', '?'), acts)
        entry['identity']['activities'] = adjusted

    save_json(data, scheduled_path)
    print(f'[OK]   {fname} — {len(data)} personnes planifiées → Temp/5_scheduled/')

print()
print('Étape 5 terminée.')

---
## Export final → `data/eqasim_output/`

Copie des fichiers planifiés depuis `Temp/5_scheduled/` vers le dossier de sortie définitif `data/eqasim_output/`. Un bilan de qualité est affiché pour chaque taille avant la copie :

| Indicateur | Signification |
|---|---|
| `missing_routes` | Agents dont au moins un trajet n'a pas pu être calculé (zones inaccessibles) |
| `missing_any_mode` | Agents dont le mode de transport reste indéterminé |
| `missing_pt` | Agents rattachés aux TC mais sans arrêt Tisséo dans le rayon de 1 500 m |

Les fichiers exportés sont directement consommables par la simulation GAMA et le serveur d'agents LLM.

In [ ]:
# ── Export final → data/eqasim_output/ ───────────────────────────────────────
from population_utils import check_enrichment

print('=' * 60)
print('EXPORT FINAL → data/eqasim_output/')
print('=' * 60)

all_ok = True
exported = []

for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname
    final_path     = POP_DIR        / fname

    if not scheduled_path.exists():
        print(f'[WARN] {fname} — Temp/5_scheduled/ absent, export ignoré (relancer étape 5)')
        all_ok = False
        continue

    data  = load_json(scheduled_path)
    stats = check_enrichment(data)
    missing = stats['missing_routes'] + stats['missing_any'] + stats['missing_pt']
    status  = 'OK' if missing == 0 else 'INCOMPLET'
    if missing > 0:
        all_ok = False

    save_json(data, final_path)
    size_mb = final_path.stat().st_size / 1_048_576
    exported.append(fname)
    print(f'[{status}] {fname}  {len(data)} personnes  {size_mb:.1f} Mo'
          f'  (missing_routes={stats["missing_routes"]},'
          f' missing_any_mode={stats["missing_any"]},'
          f' missing_pt={stats["missing_pt"]})')
    print(f'       → {final_path}')

print()
if all_ok:
    print(f'Tous les fichiers exportés avec succès ({len(exported)} taille(s)).')
else:
    print('ATTENTION : certains fichiers sont incomplets ou manquants.')
print(f'Fichiers produits : {exported}')